In [1]:
# Importing libraries
import pandas as pd 
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


def selectkbest(indep_X,dep_Y,n):
        test = SelectKBest(score_func=chi2, k=n)  # choose top n important features
        fit1= test.fit(indep_X,dep_Y)               # learn which features are important
        selectk_features = fit1.transform(indep_X)     # keep only best features
        return selectk_features                           # return reduced dataset
    
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0) # split data into train/test
        sc = StandardScaler()                               # create scaler
        X_train = sc.fit_transform(X_train)                 # learn scaling from train data and apply
        X_test = sc.transform(X_test)                       # apply same scaling to test data
        return X_train, X_test, y_train, y_test             # return all parts
    
 
def cm_prediction(classifier,X_test, y_test):    # added y_test to avoid using hidden global variable and make function independent & reusable
     y_pred = classifier.predict(X_test)         # predict output for test data
        
        # Making the Confusion Matrix
     from sklearn.metrics import confusion_matrix
     cm = confusion_matrix(y_test, y_pred)             # compare actual vs predicted
        
     from sklearn.metrics import accuracy_score 
     from sklearn.metrics import classification_report 
        #from sklearn.metrics import confusion_matrix
        #cm = confusion_matrix(y_test, y_pred)
        
     Accuracy=accuracy_score(y_test, y_pred )        # calculate accuracy
        
     report=classification_report(y_test, y_pred)      # detailed performance
     return  classifier,Accuracy,report,X_test,y_test,cm    # return results


def logistic(X_train,y_train,X_test,y_test):       
        # Fitting K-NN to the Training set
        from sklearn.linear_model import LogisticRegression                   
        classifier = LogisticRegression(random_state = 0)        # create model
        classifier.fit(X_train, y_train)                         # train model
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier, X_test, y_test)   
        return  classifier,Accuracy,report,X_test,y_test,cm      # evaluate model
    
def svm_linear(X_train,y_train,X_test,y_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'linear', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier, X_test, y_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
    
def svm_NL(X_train,y_train,X_test,y_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'rbf', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier, X_test, y_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
   
def Navie(X_train,y_train,X_test,y_test):       
        
        from sklearn.naive_bayes import GaussianNB
        classifier = GaussianNB()
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier, X_test, y_test)
        return  classifier,Accuracy,report,X_test,y_test,cm         
    
    
def knn(X_train,y_train,X_test,y_test):
           
        # Fitting K-NN to the Training set
        from sklearn.neighbors import KNeighborsClassifier
        classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier, X_test, y_test)
        return  classifier,Accuracy,report,X_test,y_test,cm

def Decision(X_train,y_train,X_test,y_test): 
 
        from sklearn.tree import DecisionTreeClassifier
        classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier, X_test, y_test)
        return  classifier,Accuracy,report,X_test,y_test,cm      


def random(X_train,y_train,X_test,y_test):
        
        from sklearn.ensemble import RandomForestClassifier
        classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier, X_test, y_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
    
def selectk_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf): 
    
    dataframe=pd.DataFrame(index=['ChiSquare'],columns=['Logistic','SVMl','SVMnl','KNN','Navie','Decision','Random'])   # create empty table
    for number,idex in enumerate(dataframe.index):          # loop through rows
        dataframe['Logistic'][idex]=acclog[number]          # fill logistic accuracy
        dataframe['SVMl'][idex]=accsvml[number]             # fill svm linear
        dataframe['SVMnl'][idex]=accsvmnl[number]           # fill svm non-linear
        dataframe['KNN'][idex]=accknn[number]               # fill knn
        dataframe['Navie'][idex]=accnav[number]             # fill naive bayesv
        dataframe['Decision'][idex]=accdes[number]          # fill decision tree
        dataframe['Random'][idex]=accrf[number]             # fill random forest
    return dataframe

In [2]:
dataset1=pd.read_csv("prep.csv",index_col=None)    # load dataset

df2=dataset1

df2 = pd.get_dummies(df2, drop_first=True)        # convert categorical to numbers

indep_X = df2.drop(columns=['classification_yes'])      # input features (X)
dep_Y=df2['classification_yes']      # target variable

In [10]:
kbest=selectkbest(indep_X,dep_Y,6)         # select top 5 features

acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

In [11]:
X_train, X_test, y_train, y_test=split_scalar(kbest,dep_Y)   
    
        
classifier,Accuracy,report,X_test,y_test,cm=logistic(X_train,y_train,X_test,y_test)   # train logistic model
acclog.append(Accuracy)   # store accuracy

classifier,Accuracy,report,X_test,y_test,cm=svm_linear(X_train,y_train,X_test,y_test)  
accsvml.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=svm_NL(X_train,y_train,X_test,y_test)  
accsvmnl.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=knn(X_train,y_train,X_test,y_test)  
accknn.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=Navie(X_train,y_train,X_test,y_test)  
accnav.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=Decision(X_train,y_train,X_test,y_test)  
accdes.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=random(X_train,y_train,X_test,y_test)  
accrf.append(Accuracy)
    
result=selectk_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)   # create comparison table

In [7]:
# k=5
result

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
ChiSquare,0.94,0.94,0.95,0.89,0.83,0.96,0.95


In [12]:
# k=6
result

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
ChiSquare,0.95,0.96,0.96,0.93,0.89,0.97,0.97
